# Sequential Recommendation with SASRec

Given a **user id**, look up everything that user has already bought, feed that purchase sequence
to a self-attentive transformer, drop the products they already own, and rank what is left.

Model: **SASRec** — *Self-Attentive Sequential Recommendation*, Kang & McAuley (ICDM 2018).
Written directly in PyTorch rather than pulled from `recbole` / `recpack`: `recpack` no longer
imports under NumPy 2, and ~120 lines of model code keeps the leave-one-out split, the
already-purchased filter and the metrics fully under our control.

In [1]:
## Imports
import json
import random
import sys
import urllib.request
from pathlib import Path

BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE))  # Go to base directory

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from src.data import Data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_colwidth", 70)

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("torch", torch.__version__, "| device:", DEVICE)

torch 2.13.0 | device: mps


In [2]:
# ---------------------------------------------------------------- configuration
# Where the purchase sequences come from:
#   "5core" -> Amazon Reviews 2023 official 5-core benchmark. Every user and every product has at
#              least 5 interactions, which is what a sequential model needs. A few small CSVs.
#   "local" -> the cached data/review/*_review.parquet files this project already downloads. Runs
#              offline, but they are the first 1,000 raw rows per category, so almost every product
#              has a single buyer and there is very little sequence to learn from.
DATA_SOURCE = "5core"
FIVE_CORE_CATEGORIES = ["All_Beauty", "Gift_Cards", "Magazine_Subscriptions"]

ENRICH_TITLES = True   # stream the product metadata dumps so recommendations have real titles
META_MAX_MB = 250      # per-category ceiling; larger dumps are skipped rather than downloaded

MIN_USER_INTERACTIONS = 3    # a user needs this many purchases before a sequence is useful
MIN_ITEM_INTERACTIONS = 2    # a product nobody else bought cannot be learned

MAX_LEN    = 50              # how many recent purchases the transformer looks at
HIDDEN_DIM = 64              # d in the paper
NUM_BLOCKS = 2               # stacked self-attention blocks
NUM_HEADS  = 1
DROPOUT    = 0.2

LR         = 1e-3
L2_EMB     = 0.0
BATCH_SIZE = 128
NUM_EPOCHS = 400
EVAL_EVERY = 10
PATIENCE   = 10              # evaluations without a val NDCG@10 gain before stopping early

NUM_EVAL_NEGS = 100          # sampled-negative protocol used in the SASRec paper
TOP_KS = (5, 10, 20)

## 1. Purchase sequences

`Data.get_review_data()` inner-joins the metadata and review tables on `parent_asin`. Only the
first 1,000 rows of each file are cached, so the two sides barely overlap and the join collapses to
~2.5k rows over ~1.9k users — hardly anyone ends up with a sequence at all. Sequential
recommendation needs *every* review, so we read the interaction data directly and treat the
metadata purely as a lookup catalogue.

In [3]:
REVIEW_DATA_DIR = BASE / "data" / "review"
FIVE_CORE_DIR = REVIEW_DATA_DIR / "5core"
HF_ROOT = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main"
FIVE_CORE_URL = HF_ROOT + "/benchmark/5core/rating_only/{category}.csv"

Data()  # a no-op once the parquet files are on disk, downloads them otherwise


def load_local_interactions(review_dir: Path) -> pd.DataFrame:
    """One row per (user, product) review event, pooled over every cached category."""
    frames = []
    for path in sorted(review_dir.glob("*_review.parquet")):
        frame = pd.read_parquet(path, columns=["user_id", "parent_asin", "rating", "timestamp"])
        frame["category"] = path.name.removesuffix("_review.parquet")
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def load_five_core_interactions(categories: list[str], cache_dir: Path) -> pd.DataFrame:
    """Official 5-core benchmark split, cached locally so it is only fetched once."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    for category in categories:
        path = cache_dir / f"{category}.csv"
        if not path.exists():
            print(f"downloading 5-core {category}...")
            pd.read_csv(FIVE_CORE_URL.format(category=category)).to_csv(path, index=False)
        frame = pd.read_csv(path)
        frame["category"] = category
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


if DATA_SOURCE == "5core":
    interactions = load_five_core_interactions(FIVE_CORE_CATEGORIES, FIVE_CORE_DIR)
elif DATA_SOURCE == "local":
    interactions = load_local_interactions(REVIEW_DATA_DIR)
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

print(f"{len(interactions):,} interactions | "
      f"{interactions.user_id.nunique():,} users | "
      f"{interactions.parent_asin.nunique():,} products")
interactions.head()

Loading ESIC DATA...
ESIC Downloaded succesfully!
Loading SQID DATA...
SQID Downloaded succesfully!
Loading Amazon Review DATA...
Amazon Review Data Downloaded succesfully!
5,140 interactions | 659 users | 506 products


,user_id,parent_asin,rating,timestamp,category
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,B07J3GH1W1,5.0,1547589356557,All_Beauty
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,B07W397QG4,5.0,1593352422858,All_Beauty
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,B07KG1TWP5,5.0,1596473351088,All_Beauty
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,B08JTNQFZY,5.0,1617904219785,All_Beauty
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,B07SLFWZKN,3.0,1619737501209,All_Beauty


In [4]:
def k_core_filter(frame: pd.DataFrame, min_user: int, min_item: int) -> pd.DataFrame:
    """Drop rare users and rare products in turn until both thresholds hold at the same time."""
    frame = frame.sort_values("timestamp", kind="stable")
    frame = frame.drop_duplicates(["user_id", "parent_asin"], keep="first")  # keep the first purchase
    while not frame.empty:
        before = len(frame)
        frame = frame[frame.groupby("user_id")["user_id"].transform("size") >= min_user]
        frame = frame[frame.groupby("parent_asin")["parent_asin"].transform("size") >= min_item]
        if len(frame) == before:
            break
    return frame.sort_values(["user_id", "timestamp"], kind="stable").reset_index(drop=True)


core = k_core_filter(interactions, MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS)

lengths = core.user_id.value_counts()
print(f"after {MIN_USER_INTERACTIONS}-core users / {MIN_ITEM_INTERACTIONS}-core products:")
print(f"  {len(core):,} interactions | {core.user_id.nunique():,} users | {core.parent_asin.nunique():,} products")
print(f"  sequence length: mean {lengths.mean():.1f}, median {lengths.median():.0f}, max {lengths.max()}")
print(f"  density: {len(core) / (core.user_id.nunique() * core.parent_asin.nunique()):.3%}")

after 3-core users / 2-core products:
  5,140 interactions | 659 users | 506 products
  sequence length: mean 7.8, median 6, max 63
  density: 1.541%


In [5]:
# Contiguous ids for the embedding table; 0 stays free as the padding id.
item_to_index = {asin: index for index, asin in enumerate(core.parent_asin.unique(), start=1)}
index_to_item = {index: asin for asin, index in item_to_index.items()}
NUM_ITEMS = len(item_to_index)

core["item_index"] = core.parent_asin.map(item_to_index)
user_sequences = core.groupby("user_id", sort=False)["item_index"].apply(list).to_dict()

print(f"{NUM_ITEMS:,} products | {len(user_sequences):,} user sequences")

506 products | 659 user sequences


### Product catalogue

The cached `*_meta.parquet` files are also the first 1,000 rows per category, so they cover a
different slice of the catalogue than the reviews do and almost none of our products get a title.
Streaming the full metadata dump for the categories in play fixes that; it is a one-off pass that
is cached as a small parquet, and any dump larger than `META_MAX_MB` is skipped instead.

In [6]:
META_URL = HF_ROOT + "/raw/meta_categories/meta_{category}.jsonl"
META_CACHE_DIR = REVIEW_DATA_DIR / "titles"
CATALOG_COLUMNS = ["parent_asin", "title", "main_category", "store", "price", "average_rating"]


def remote_size_mb(url: str) -> float:
    with urllib.request.urlopen(urllib.request.Request(url, method="HEAD")) as response:
        headers = response.headers
        return int(headers.get("x-linked-size") or headers["content-length"]) / 1e6


def fetch_category_titles(category: str, cache_dir: Path, max_mb: float) -> pd.DataFrame | None:
    """Stream one product metadata dump and keep only the columns worth displaying."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f"{category}.parquet"
    if not path.exists():
        size_mb = remote_size_mb(META_URL.format(category=category))
        if size_mb > max_mb:
            return None
        print(f"  streaming {size_mb:,.0f} MB of {category} metadata (cached afterwards)...")
        records = []
        with urllib.request.urlopen(META_URL.format(category=category)) as response:
            for line in response:
                record = json.loads(line)
                records.append({column: record.get(column) for column in CATALOG_COLUMNS})
        frame = pd.DataFrame(records, columns=CATALOG_COLUMNS)
        frame["price"] = frame["price"].astype("string")
        frame.to_parquet(path, index=False)
    return pd.read_parquet(path)


def build_catalog(review_dir: Path, categories, enrich: bool, max_mb: float) -> pd.DataFrame:
    """Cached metadata first, freshly streamed metadata last so it wins on duplicates."""
    frames = [pd.read_parquet(path, columns=CATALOG_COLUMNS)
              for path in sorted(review_dir.glob("*_meta.parquet"))]
    if enrich:
        print("product metadata:")
        skipped = []
        for category in categories:
            frame = fetch_category_titles(category, META_CACHE_DIR, max_mb)
            if frame is None:
                skipped.append(category)
            else:
                frames.append(frame)
        if skipped:
            print(f"  skipped (dump larger than META_MAX_MB={max_mb:g}): {', '.join(skipped)}")

    catalog = pd.concat(frames, ignore_index=True)
    catalog["price"] = catalog["price"].astype("string")
    return (catalog.dropna(subset=["parent_asin"])
                   .drop_duplicates("parent_asin", keep="last")
                   .set_index("parent_asin"))


catalog = build_catalog(REVIEW_DATA_DIR, sorted(core.category.unique()), ENRICH_TITLES, META_MAX_MB)
covered = catalog.index.intersection(core.parent_asin.unique())
print(f"{len(catalog):,} products in the catalogue | titles for {len(covered) / NUM_ITEMS:.1%} of the products we recommend")

product metadata:
146,759 products in the catalogue | titles for 100.0% of the products we recommend


## 2. Leave-one-out split

The standard protocol for sequential recommendation: each user's newest purchase is the test
target, the one before it is validation, everything earlier is training. The split is per user and
chronological, so the model is never shown a purchase that happens after the one it must predict.

In [7]:
def leave_one_out(sequences: dict[str, list[int]]):
    train, valid, test = {}, {}, {}
    for user, sequence in sequences.items():
        train[user] = sequence[:-2]
        valid[user] = sequence[-2]
        test[user] = sequence[-1]
    return train, valid, test


train_sequences, valid_targets, test_targets = leave_one_out(user_sequences)

# Validation predicts from the training prefix; the test step may also use the validation purchase,
# because that one happened before the test purchase.
valid_inputs = train_sequences
test_inputs = {user: sequence + [valid_targets[user]] for user, sequence in train_sequences.items()}

train_items = {item for sequence in train_sequences.values() for item in sequence}
coverage = np.mean([test_targets[user] in train_items for user in test_targets])
print(f"train interactions: {sum(len(s) for s in train_sequences.values()):,}")
print(f"test targets seen during training: {coverage:.1%}  <- ceiling on full-catalogue recall")

train interactions: 3,822
test targets seen during training: 89.5%  <- ceiling on full-catalogue recall


## 3. The model

Each block is `LayerNorm -> causal multi-head self-attention -> residual -> LayerNorm ->
point-wise feed-forward -> residual`, the pre-norm arrangement of the reference implementation.
The causal mask is what makes it *sequential*: position `t` may only attend to positions `<= t`,
so the hidden state sitting after the newest purchase is a prediction of the next one.

Scoring is a dot product against the **shared item embedding table** — recommending means
comparing the user's current state against every product vector at once.

In [8]:
class PointWiseFeedForward(nn.Module):
    """Two 1x1 convolutions applied independently at every position, with a residual."""

    def __init__(self, hidden_dim: int, dropout: float):
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, inputs):
        hidden = inputs.transpose(-1, -2)
        hidden = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(hidden)))))
        return hidden.transpose(-1, -2) + inputs


class SASRec(nn.Module):
    """Self-Attentive Sequential Recommendation (Kang & McAuley, 2018)."""

    def __init__(self, num_items, hidden_dim, max_len, num_blocks, num_heads, dropout):
        super().__init__()
        self.num_items = num_items
        self.max_len = max_len
        self.hidden_dim = hidden_dim

        self.item_emb = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_dim)
        self.emb_dropout = nn.Dropout(dropout)

        self.attention_layernorms = nn.ModuleList()
        self.attention_layers = nn.ModuleList()
        self.forward_layernorms = nn.ModuleList()
        self.forward_layers = nn.ModuleList()
        for _ in range(num_blocks):
            self.attention_layernorms.append(nn.LayerNorm(hidden_dim, eps=1e-8))
            self.attention_layers.append(
                nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
            )
            self.forward_layernorms.append(nn.LayerNorm(hidden_dim, eps=1e-8))
            self.forward_layers.append(PointWiseFeedForward(hidden_dim, dropout))
        self.last_layernorm = nn.LayerNorm(hidden_dim, eps=1e-8)

        for name, parameter in self.named_parameters():
            if parameter.dim() > 1:
                nn.init.xavier_normal_(parameter)
        with torch.no_grad():
            self.item_emb.weight[0].zero_()

    def encode(self, sequences):
        """sequences: (batch, max_len) item ids, left padded with 0 -> (batch, max_len, hidden)."""
        hidden = self.item_emb(sequences) * (self.hidden_dim ** 0.5)
        positions = torch.arange(self.max_len, device=sequences.device).expand_as(sequences)
        hidden = self.emb_dropout(hidden + self.pos_emb(positions))

        # Padded positions are zeroed rather than masked out of attention: under a causal mask a
        # fully padded query row would softmax over nothing and produce NaNs.
        padding = (sequences == 0).unsqueeze(-1)
        hidden = hidden.masked_fill(padding, 0.0)

        causal_mask = torch.ones(self.max_len, self.max_len, device=sequences.device).triu(1).bool()
        blocks = zip(self.attention_layernorms, self.attention_layers,
                     self.forward_layernorms, self.forward_layers)
        for attention_norm, attention, forward_norm, feed_forward in blocks:
            queries = attention_norm(hidden)
            attended, _ = attention(queries, hidden, hidden, attn_mask=causal_mask, need_weights=False)
            hidden = queries + attended
            hidden = feed_forward(forward_norm(hidden))
            hidden = hidden.masked_fill(padding, 0.0)

        return self.last_layernorm(hidden)

    def forward(self, sequences, positives, negatives):
        """Training scores: one positive and one sampled negative at every position."""
        hidden = self.encode(sequences)
        positive_logits = (hidden * self.item_emb(positives)).sum(dim=-1)
        negative_logits = (hidden * self.item_emb(negatives)).sum(dim=-1)
        return positive_logits, negative_logits

    def score_all_items(self, sequences):
        """State after the newest purchase, scored against the whole catalogue."""
        state = self.encode(sequences)[:, -1, :]
        return state @ self.item_emb.weight.T

## 4. Training

One training example per user per epoch: the sequence shifted by one position, so every prefix
predicts the purchase that follows it. Each positive is paired with a uniformly sampled product the
user has never touched, and the loss is binary cross-entropy over both — the objective from the
paper. Padded positions are excluded from the loss.

In [9]:
def pad_sequence(sequence: list[int], max_len: int) -> np.ndarray:
    """Right-align a sequence so the newest purchase always sits in the last position."""
    padded = np.zeros(max_len, dtype=np.int64)
    window = sequence[-max_len:]
    if window:
        padded[max_len - len(window):] = window
    return padded


def sample_negative(seen: set[int], num_items: int, rng: np.random.Generator) -> int:
    """A uniformly random product the user has not interacted with."""
    for _ in range(100):
        candidate = int(rng.integers(1, num_items + 1))
        if candidate not in seen:
            return candidate
    return candidate


class SequenceDataset(Dataset):
    """(input, next-purchase, sampled-negative) triples, one row per user."""

    def __init__(self, sequences, num_items, max_len):
        self.users = [user for user, sequence in sequences.items() if len(sequence) >= 2]
        self.sequences = sequences
        self.num_items = num_items
        self.max_len = max_len
        self.rng = np.random.default_rng(SEED)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, index):
        sequence = self.sequences[self.users[index]]
        seen = set(sequence)
        inputs = pad_sequence(sequence[:-1], self.max_len)
        positives = pad_sequence(sequence[1:], self.max_len)
        negatives = np.where(
            positives > 0,
            [sample_negative(seen, self.num_items, self.rng) for _ in range(self.max_len)],
            0,
        ).astype(np.int64)
        return inputs, positives, negatives


train_dataset = SequenceDataset(train_sequences, NUM_ITEMS, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"{len(train_dataset):,} trainable users | {len(train_loader)} batches per epoch")

659 trainable users | 6 batches per epoch


### Evaluation

Two protocols, because they answer different questions:

* **Sampled negatives (100)** — the protocol from the SASRec paper: rank the held-out purchase
  against 100 random products the user never bought. Comparable to published numbers.
* **Full catalogue** — rank it against *every* product the user has not bought. This is what the
  deployed recommender actually does, and it is much harder.

A popularity baseline goes through the identical code path, so the transformer's numbers mean
something rather than just looking large.

In [10]:
def ranking_metrics(ranks: np.ndarray, ks=TOP_KS) -> dict[str, float]:
    """ranks are 0-based positions of the held-out purchase."""
    metrics = {}
    for k in ks:
        hit = ranks < k
        metrics[f"HR@{k}"] = float(hit.mean())
        metrics[f"NDCG@{k}"] = float(np.where(hit, 1.0 / np.log2(ranks + 2.0), 0.0).mean())
    return metrics


@torch.no_grad()
def evaluate(score_fn, inputs, targets, histories, num_negatives=None, ks=TOP_KS, seed=SEED, batch_size=256):
    """score_fn: (batch, max_len) padded sequences -> (batch, num_items + 1) product scores.

    num_negatives=None ranks against the full catalogue, otherwise against that many sampled
    negatives. Products the user already bought never compete with the held-out target.
    """
    rng = np.random.default_rng(seed)
    users = [user for user in targets if inputs.get(user)]
    ranks = []

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        sequences = torch.as_tensor(
            np.stack([pad_sequence(inputs[user], MAX_LEN) for user in batch_users])
        ).to(DEVICE)
        scores = score_fn(sequences).float().cpu().numpy()
        scores[:, 0] = -np.inf  # the padding id is not a product

        for row, user in enumerate(batch_users):
            target = targets[user]
            history = histories[user] - {target}
            if num_negatives is None:
                row_scores = scores[row].copy()
                row_scores[list(history)] = -np.inf
                ranks.append(int((row_scores > row_scores[target]).sum()))
            else:
                forbidden = history | {target}
                budget = min(num_negatives, NUM_ITEMS - len(forbidden))
                negatives, taken = [], set(forbidden)
                while len(negatives) < budget:
                    for value in rng.integers(1, NUM_ITEMS + 1, size=budget * 2):
                        if int(value) not in taken:
                            taken.add(int(value))
                            negatives.append(int(value))
                            if len(negatives) == budget:
                                break
                candidates = [target] + negatives
                candidate_scores = scores[row][candidates]
                ranks.append(int((candidate_scores > candidate_scores[0]).sum()))

    return ranking_metrics(np.asarray(ranks), ks), len(ranks)


histories = {user: set(sequence) for user, sequence in user_sequences.items()}


def model_scorer(model):
    def score(sequences):
        model.eval()
        return model.score_all_items(sequences)
    return score

In [11]:
model = SASRec(NUM_ITEMS, HIDDEN_DIM, MAX_LEN, NUM_BLOCKS, NUM_HEADS, DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98))
criterion = nn.BCEWithLogitsLoss(reduction="none")

best_ndcg, best_state, evaluations_without_gain = -1.0, None, 0
training_log = []

progress = tqdm(range(1, NUM_EPOCHS + 1), desc="training")
for epoch in progress:
    model.train()
    epoch_loss = 0.0
    for sequences, positives, negatives in train_loader:
        sequences = sequences.to(DEVICE)
        positives = positives.to(DEVICE)
        negatives = negatives.to(DEVICE)

        positive_logits, negative_logits = model(sequences, positives, negatives)
        mask = (positives != 0).float()
        loss = (criterion(positive_logits, torch.ones_like(positive_logits)) * mask).sum() / mask.sum()
        loss = loss + (criterion(negative_logits, torch.zeros_like(negative_logits)) * mask).sum() / mask.sum()
        if L2_EMB:
            loss = loss + L2_EMB * model.item_emb.weight.norm(2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if epoch % EVAL_EVERY == 0:
        scores, _ = evaluate(model_scorer(model), valid_inputs, valid_targets, histories,
                             num_negatives=NUM_EVAL_NEGS)
        training_log.append({"epoch": epoch, "loss": epoch_loss / len(train_loader), **scores})
        progress.set_postfix(loss=f"{epoch_loss / len(train_loader):.4f}",
                             val_ndcg10=f"{scores['NDCG@10']:.4f}")

        if scores["NDCG@10"] > best_ndcg:
            best_ndcg = scores["NDCG@10"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            evaluations_without_gain = 0
        else:
            evaluations_without_gain += 1
            if evaluations_without_gain >= PATIENCE:
                progress.close()
                print(f"early stop at epoch {epoch}")
                break

model.load_state_dict(best_state)
print(f"best validation NDCG@10: {best_ndcg:.4f}")
pd.DataFrame(training_log).set_index("epoch").tail(8).round(4)

training:   0%|          | 0/400 [00:00<?, ?it/s]

early stop at epoch 200
best validation NDCG@10: 0.4322


,loss,HR@5,NDCG@5,HR@10,NDCG@10,HR@20,NDCG@20
epoch,,,,,,,
130,0.3285,0.5068,0.3724,0.6480,0.4183,0.8118,0.4594
140,0.3076,0.5311,0.3850,0.6464,0.4221,0.8179,0.4653
150,0.2873,0.5114,0.3816,0.6540,0.4277,0.8088,0.4665
160,0.2877,0.4992,0.3655,0.6358,0.4099,0.7951,0.4500
170,0.2784,0.5205,0.3870,0.6434,0.4270,0.8042,0.4677
180,0.2516,0.5038,0.3745,0.6495,0.4217,0.8058,0.4610
190,0.2423,0.4917,0.3710,0.6449,0.4211,0.7921,0.4582
200,0.2265,0.4871,0.3691,0.6419,0.4199,0.7936,0.4582


In [12]:
popularity = np.zeros(NUM_ITEMS + 1, dtype=np.float32)
for sequence in train_sequences.values():
    for item in sequence:
        popularity[item] += 1


def popularity_scorer(sequences):
    """Ignores the sequence entirely — the honest "recommend the bestsellers" baseline."""
    counts = torch.as_tensor(popularity, device=sequences.device)
    return counts.unsqueeze(0).repeat(sequences.shape[0], 1)


rows = []
for name, scorer in [("popularity", popularity_scorer), ("SASRec", model_scorer(model))]:
    for protocol, negatives in [(f"sampled ({NUM_EVAL_NEGS} negs)", NUM_EVAL_NEGS), ("full catalogue", None)]:
        scores, evaluated = evaluate(scorer, test_inputs, test_targets, histories, num_negatives=negatives)
        rows.append({"protocol": protocol, "model": name, "users": evaluated, **scores})

results = pd.DataFrame(rows).set_index(["protocol", "model"]).sort_index()
results.round(4)

users    HR@5  NDCG@5   HR@10  NDCG@10   HR@20  \
protocol           model                                                        
full catalogue     SASRec        659  0.2413  0.1650  0.3429   0.1983  0.4522   
                   popularity    659  0.1517  0.1101  0.2458   0.1403  0.3020   
sampled (100 negs) SASRec        659  0.4704  0.3586  0.5903   0.3975  0.7269   
                   popularity    659  0.3232  0.2403  0.4173   0.2707  0.4841   

                               NDCG@20  
protocol           model                
full catalogue     SASRec       0.2261  
                   popularity   0.1542  
sampled (100 negs) SASRec       0.4319  
                   popularity   0.2875

## 5. Recommending for a user id

This is what the problem statement asks for: hand it a user id, it pulls that user's purchase
history, runs the sequence through SASRec, **removes everything they have already bought**, and
returns the highest scoring products left in the catalogue.

A user the model has never seen falls back to a popularity ranking instead of failing.

In [13]:
DISPLAY_COLUMNS = ["rank", "title", "main_category", "store", "price", "average_rating", "score", "parent_asin", "url"]


def describe(item_indices, scores=None) -> pd.DataFrame:
    """Turn internal item ids into a readable table using the product catalogue."""
    asins = [index_to_item[index] for index in item_indices]
    frame = catalog.reindex(asins).reset_index()
    frame.insert(0, "rank", range(1, len(frame) + 1))
    frame["title"] = frame["title"].fillna("(no metadata available)")
    frame["url"] = "https://www.amazon.com/dp/" + frame["parent_asin"]
    frame["score"] = np.asarray(scores) if scores is not None else np.nan
    return frame[DISPLAY_COLUMNS]


def purchase_history(user_id: str) -> pd.DataFrame:
    """Everything this user has bought, oldest first."""
    bought = core[core.user_id == user_id].sort_values("timestamp")
    frame = bought[["parent_asin", "category", "rating", "timestamp"]].copy()
    frame["purchased_at"] = pd.to_datetime(frame.pop("timestamp"), unit="ms").dt.date
    frame = frame.merge(catalog[["title", "store"]], how="left", left_on="parent_asin", right_index=True)
    frame["title"] = frame["title"].fillna("(no metadata available)")
    return frame[["purchased_at", "title", "category", "store", "rating", "parent_asin"]].reset_index(drop=True)


@torch.no_grad()
def recommend(user_id: str, k: int = 10, exclude_purchased: bool = True) -> pd.DataFrame:
    """Top-k products for one user id, ranked by SASRec over their purchase sequence."""
    sequence = user_sequences.get(user_id)
    if not sequence:
        print(f"user {user_id!r} has no purchase history here — falling back to popularity")
        scores = torch.as_tensor(popularity, device=DEVICE).clone()
        sequence = []
    else:
        model.eval()
        padded = torch.as_tensor(pad_sequence(sequence, MAX_LEN)).unsqueeze(0).to(DEVICE)
        scores = model.score_all_items(padded).squeeze(0)

    scores[0] = -np.inf  # padding id
    if exclude_purchased and sequence:
        scores[torch.as_tensor(sequence, device=DEVICE)] = -np.inf  # <- drop what he already bought

    ranked = torch.topk(scores, min(k + len(sequence) + 1, NUM_ITEMS + 1)).indices.cpu().numpy()
    top = [int(index) for index in ranked if int(index) in index_to_item][:k]
    return describe(top, scores[top].cpu().numpy())


# Users with the richest histories make the clearest demo.
candidate_users = sorted(user_sequences, key=lambda user: len(user_sequences[user]), reverse=True)
pd.DataFrame({
    "user_id": candidate_users[:10],
    "purchases": [len(user_sequences[user]) for user in candidate_users[:10]],
})

,user_id,purchases
0,AG73BVBKUOH22USSFJA5ZWL7AKXA,63
1,AEZP6Z2C5AVQDZAJECQYZWQRNG3Q,58
2,AEOK4TQIKGO23SJKZ6PW4FETNNDA_1,36
3,AEMP3A7IKW37CMWFXNKXWW6HGJHA_1,34
4,AHY2TURQPNIDXZGH2CMQLZ343YMQ,34
5,AF4OCZTPFHXZKUTAGP6CAOCMWPAA,32
6,AGYVC7KVHP2AWM7BDCEYNHFA6F3Q,30
7,AF2BLE54TEMGZ546U763ZHZRXC4A,29
8,AGZUJTI7A3JFKB4FP5JOH6NVAJIQ_1,28
9,AH3BXW7KLIS2VAE56UXJS2NS7I5A,28


In [14]:
USER_ID = candidate_users[0]   # <- put any user id from the table above here
# USER_ID = input("user id: ").strip()   # uncomment to type one in instead

print(f"purchase history for {USER_ID}")
purchase_history(USER_ID).tail(10)

purchase history for AG73BVBKUOH22USSFJA5ZWL7AKXA


,purchased_at,title,category,store,rating,parent_asin
53,2021-04-06,OGANA CELL Peptide Concentrating Amazing Lotion 2.03 fl.oz. (60ml)...,All_Beauty,OGANA CELL,5.0,B07RBSLNFR
54,2021-04-14,BABOR Power Serum Ampoules - Hyaluronic Acid | Moisturizing Smooth...,All_Beauty,None,5.0,B08VQY1Z1S
55,2021-05-07,CELIMAX The Real Sedum Aqua Boosting Essence - with 65.33% Sedum E...,All_Beauty,CELIMAX,5.0,B084WP4XS8
56,2021-05-21,ZOUSZ Black Oud Vegan Beard Balm - Classic Wood-Scented Facial Hai...,All_Beauty,ZOUSZ,5.0,B08BKWLHYP
57,2021-05-21,"HASK CUCUMBER + ALOE Body Wash 2 Pack- Refreshing, vegan formula, ...",All_Beauty,HASK,5.0,B08SG2MBRY
58,2021-05-24,NIRA Skincare Laser & Serum Bundle - Includes Anti-Aging Laser & H...,All_Beauty,Nira,5.0,B08P2DZB4X
59,2021-06-18,AAVRANI Travel Discovery Skin Care Set with Exfoliating Cleanser &...,All_Beauty,AAVRANI,5.0,B0949MJRHK
60,2021-06-22,"Burt's Bees Baby Bath Time Kit with Ultra Gentle Lotion, Bubble Ba...",All_Beauty,Burt's Bees,5.0,B08MC3ZLV4
61,2021-07-10,"2 Pack Foundation Full Coverage Liquid Foundation, Foundation Prim...",All_Beauty,Onkessy,4.0,B08SBV6Z57
62,2021-09-24,Rated Green - Hibiscus Moisturizing Scalp Pack - Tube - 6.76 fl. o...,All_Beauty,RATED GREEN,4.0,B083PXJBVY


In [15]:
print(f"recommendations for {USER_ID} (already-purchased products filtered out)")
recommend(USER_ID, k=10)

recommendations for AG73BVBKUOH22USSFJA5ZWL7AKXA (already-purchased products filtered out)


,rank,title,main_category,store,price,average_rating,score,parent_asin,url
0,1,"Spanature Olive Hand & Body Lotion Travel Size Selection, 4 Pack, ...",All Beauty,Spanature,<NA>,4.8,2.923214,B01M24DUXH,https://www.amazon.com/dp/B01M24DUXH
1,2,Sand & Sky Face Roller & Massager Tool. Aussie Skincare Essentials...,All Beauty,Sand & Sky,<NA>,4.8,2.900270,B08RYN11N9,https://www.amazon.com/dp/B08RYN11N9
2,3,"Bloomeffects - Natural Tulip Dew Face Cream | Cruelty-Free, Non-To...",All Beauty,Bloomeffects,<NA>,3.2,2.524146,B09FNTZFZZ,https://www.amazon.com/dp/B09FNTZFZZ
3,4,Lemon Oil Towelettes - 20 Count,All Beauty,Herban Essentials,16.0,4.9,2.517189,B096S4RQNH,https://www.amazon.com/dp/B096S4RQNH
4,5,C'est Moi Vanilla Vibes Sugar Balm Body Scrub | Invigorating & Hyd...,All Beauty,C'est Moi,20.5,3.7,2.285181,B07FPS2VFK,https://www.amazon.com/dp/B07FPS2VFK
5,6,"BeautyStat Cosmetics Universal Moisture Essence, 100% Pure Squalan...",All Beauty,BeautyStat,40.0,4.5,2.159267,B08RRSPNWV,https://www.amazon.com/dp/B08RRSPNWV
6,7,TINY BUDDHA Mindfulness Gift Set Includes Soothing Bath and Shower...,All Beauty,Tiny Buddha,<NA>,4.5,2.046338,B08P5579KC,https://www.amazon.com/dp/B08P5579KC
7,8,"UNNY CLUB Cover Glow Cushion 0.3 Oz / 11g Beige Color SPF50+, Flaw...",All Beauty,UNNY CLUB,<NA>,3.7,1.972776,B08K2V6WGH,https://www.amazon.com/dp/B08K2V6WGH
8,9,"Spanature Aloe Vera Hand & Body Lotion Travel Size Selection, 4 Pa...",All Beauty,Spanature,<NA>,3.9,1.964177,B08LPGZMQK,https://www.amazon.com/dp/B08LPGZMQK
9,10,YERMA Makeup Face Primer with Brush - Instant Brightening and Whit...,All Beauty,Yerma,<NA>,3.8,1.823118,B0977LVHF8,https://www.amazon.com/dp/B0977LVHF8


In [16]:
# Sanity check: nothing recommended is something the user already owns.
for user in candidate_users[:25]:
    recommended = set(recommend(user, k=20).parent_asin)
    assert not recommended & set(core[core.user_id == user].parent_asin), user
print("checked 25 users: no recommendation overlaps a past purchase")

checked 25 users: no recommendation overlaps a past purchase


## 6. Saving the trained recommender

In [17]:
ARTIFACT_DIR = Path.cwd() / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

torch.save(
    {
        "state_dict": model.state_dict(),
        "item_to_index": item_to_index,
        "config": {
            "num_items": NUM_ITEMS,
            "hidden_dim": HIDDEN_DIM,
            "max_len": MAX_LEN,
            "num_blocks": NUM_BLOCKS,
            "num_heads": NUM_HEADS,
            "dropout": DROPOUT,
            "data_source": DATA_SOURCE,
        },
    },
    ARTIFACT_DIR / "sasrec.pt",
)
print("saved", ARTIFACT_DIR / "sasrec.pt")

saved /Users/rohit/Multi-Modal-Product-Search-and-Recommendation-System/src/recommender/artifacts/sasrec.pt
